In [29]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [21]:
# turning big numbers into 0. numbers, for model to understand better
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [22]:
# splitting the training data into training and validation sets
rng = np.random.default_rng(42)
idx = rng.permutation(len(x_train))

x_val, y_val = x_train[idx[:10000]], y_train[idx[:10000]]
x_train, y_train = x_train[idx[10000:]], y_train[idx[10000:]]

print(x_train.shape, y_train.shape)

(50000, 28, 28) (50000,)


In [23]:
# reshaping the data to be 2D array for the model
x_train_flat = x_train.reshape(-1, 28*28)
x_val_flat = x_val.reshape(-1, 28*28)
x_test_flat = x_test.reshape(-1, 28*28)

# converting the labels to int64 for compatibility with TensorFlow
y_train = y_train.astype('int64')
y_val = y_val.astype('int64')
y_test = y_test.astype('int64')

In [24]:
tf.random.set_seed(42)
# initializing weights and biases
W1 = tf.Variable(tf.random.normal((784, 128)) * (2 / 784) ** 0.5)
#128 zeros for each of the 128 neurons in the hidden layer
b1 = tf.Variable(tf.zeros(128))
W2 = tf.Variable(tf.random.normal((128, 10)) * (2 / 128) ** 0.5)
b2 = tf.Variable(tf.zeros(10))
params = [W1, b1, W2, b2]

In [25]:
# PART 3: THE NETWORK'S BUILDING BLOCKS
# Three functions we will use again and again:
#   forward     -> runs images through the network and gets 10 scores per image
#   loss_fn     -> measures how wrong those scores are (one number, lower = better)
#   accuracy_fn -> measures what fraction of images the network gets right

# --- forward: images in, scores out ---
# Python syntax:
#   def name(arguments):  -> defines a function (like `function name(arguments) {}` in JS)
#   The indented lines below are the function's body. Python uses indentation instead of { }.
#   x is the argument: a batch of flattened images, shape (n, 784) where n = number of images
def forward(x):
    # Hidden layer, step 1: multiply the pixels by W1 and add the bias b1.
    # tf.matmul(a, b) -> matrix multiplication: (n, 784) x (784, 128) = (n, 128)
    # + b1            -> adds b1 (128 numbers) to every one of the n rows ("broadcasting")
    # z1 shape: (n, 128) = 128 raw scores for each image
    # W1 and b1 are not arguments: the function reads them from the earlier cell.
    # A function can use variables defined outside it (they are "global" in the notebook).
    z1 = tf.matmul(x, W1) + b1

    # Hidden layer, step 2: ReLU turns every negative number into 0, keeps positives as they are.
    # e.g. [-2.1, 0.5, 3.0] -> [0, 0.5, 3.0]
    # Without this step, two layers would be no smarter than one.
    # h shape: (n, 128)
    h = tf.nn.relu(z1)

    # Output layer: multiply by W2 and add b2. (n, 128) x (128, 10) = (n, 10)
    # These 10 numbers per image are the "logits": raw scores, one per digit 0-9.
    # The highest score is the network's guess.
    # `return` sends the result back to whoever called the function.
    return tf.matmul(h, W2) + b2


# --- loss_fn: how wrong are the scores? ---
# logits: the (n, 10) scores from forward()
# y:      the correct labels, shape (n,), e.g. [5, 0, 4, ...]
def loss_fn(logits, y):
    # Read this line from the inside out:
    # 1. tf.nn.sparse_softmax_cross_entropy_with_logits(...)
    #      - turns each image's 10 scores into 10 probabilities (softmax)
    #      - takes the probability given to the correct digit and computes -log(probability)
    #        e.g. 90% sure of the right digit -> small loss, 5% sure -> big loss
    #      - "sparse" = labels are plain digits like 3, not one-hot [0,0,0,1,0,...]
    #      - result: one loss number per image, shape (n,)
    # 2. tf.reduce_mean(...) -> average of those n numbers -> one single number
    # Python syntax:
    #   labels=y, logits=logits -> "keyword arguments": passing values by name.
    #   The name on the left is the function's parameter, the value on the right is our variable.
    return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=logits))


# --- accuracy_fn: what fraction of images does the network get right? ---
# x: images, shape (n, 784)
# y: correct labels, shape (n,)
def accuracy_fn(x, y):
    # forward(x)          -> scores, shape (n, 10)
    # tf.argmax(..., axis=1) -> for each image, the POSITION of the highest score = the guessed digit
    #                          e.g. scores [0.1, 0.2, 5.3, 0.4, ...] -> 2
    #   axis=1 means "look along each row" (across the 10 digits), not down the columns
    # preds shape: (n,) = one guessed digit per image
    preds = tf.argmax(forward(x), axis=1)

    # Read from the inside out:
    # 1. tf.equal(preds, y)        -> True where the guess is right, False where wrong
    #                                 e.g. [True, False, True, True]
    # 2. tf.cast(..., tf.float32)  -> convert True/False to numbers 1.0/0.0
    #                                 e.g. [1.0, 0.0, 1.0, 1.0]
    # 3. tf.reduce_mean(...)       -> the average = fraction correct, e.g. 0.75 = 75%
    # 4. .numpy()                  -> turn the TensorFlow tensor into a plain number for printing
    # Python syntax:
    #   a.b().c() -> "method chaining": call one thing after another on the result
    return tf.reduce_mean(tf.cast(tf.equal(preds, y), tf.float32)).numpy()


# Try it on the validation set before any training.
# The weights are still random, so this should be about 0.1 (10%), the same as guessing
# one of 10 digits at random. After training it should be about 0.97.
print(accuracy_fn(x_val_flat, y_val))

0.117


In [26]:
# PART 4: UNDERSTANDING CHECK
# Goal: calculate the gradient of W2 two ways (TensorFlow automatically, and by hand
# with a formula) and check that both give the same answer.
# Gradient = for every weight: "if I nudge it up a tiny bit, does the loss go up or down, and how much?"

# Take the first 64 images and their 64 labels as one small batch.
# Python syntax:
#   a, b = 1, 2      -> "multiple assignment": a gets 1, b gets 2, in one line
#   arr[:64]         -> "slicing": items from index 0 up to (not including) 64
# xb shape: (64, 784)  = 64 images, 784 pixels each
# yb shape: (64,)      = 64 labels, e.g. [5, 0, 4, ...]
xb, yb = x_train_flat[:64], y_train[:64]

# Start recording. Every calculation inside this block is written down by the "tape",
# so TensorFlow can later play it backwards and work out the gradients.
# Python syntax:
#   with X as name:  -> a "context manager": X is switched on at the start of the block
#                       and switched off automatically at the end. Here it starts and
#                       stops the recording. `tape` is the name we give the recorder.
#   The indented lines below belong to the block. Python uses indentation instead of { }.
with tf.GradientTape() as tape:
    # Hidden layer, step 1: multiply the pixels by W1 and add the bias b1.
    # tf.matmul(a, b) -> matrix multiplication: (64, 784) x (784, 128) = (64, 128)
    # + b1            -> adds b1 (128 numbers) to every one of the 64 rows ("broadcasting")
    # z1 shape: (64, 128) = 128 raw scores for each of the 64 images
    z1 = tf.matmul(xb, W1) + b1

    # Hidden layer, step 2: ReLU turns every negative number into 0, keeps positives as they are.
    # h is "how active" each of the 128 hidden units is for each image.
    # h shape: (64, 128)
    h = tf.nn.relu(z1)

    # Output layer: multiply the hidden values by W2 and add b2.
    # (64, 128) x (128, 10) = (64, 10) -> 10 raw scores ("logits") per image, one per digit 0-9
    logits = tf.matmul(h, W2) + b2

    # Loss: one number saying how wrong the model is on these 64 images (lower = better).
    # loss_fn is the function we defined in the previous cell.
    loss = loss_fn(logits, yb)

# --- Way 1: TensorFlow calculates the gradient ---
# This line is NOT indented, so it is outside the `with` block: recording has stopped.
# tape.gradient(what, with_respect_to) plays the recording backwards and answers:
# "how much does `loss` change if each number in W2 changes a tiny bit?"
# dW2_tape shape: (128, 10) = same shape as W2, one gradient per weight
dW2_tape = tape.gradient(loss, W2)

# --- Way 2 (switched off): the same gradient by hand ---
# Commented out: training only uses Way 1. To run the check again, remove the "# "
# in front of the code lines below (select them and press Cmd+/).

# Turn the 10 raw scores per image into 10 probabilities that add up to 1.
# e.g. [0.01, 0.02, 0.05, 0.80, ...] = "80% sure it's a 3"
# probs shape: (64, 10)
# probs = tf.nn.softmax(logits)

# Write each correct label as 10 numbers: a 1 at the right digit, 0 everywhere else.
# tf.one_hot(labels, 10) -> label 3 becomes [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
# onehot shape: (64, 10)
# onehot = tf.one_hot(yb, 10)

# The formula: blame for each weight = how active its input was x how wrong its output was.
#   probs - onehot        -> how wrong each guess was, shape (64, 10)
#   transpose_a=True      -> flip h from (64, 128) to (128, 64) before multiplying ("h.T")
#   tf.matmul(...)        -> (128, 64) x (64, 10) = (128, 10), same shape as W2
#   / 64                  -> average over the 64 images in the batch
# Python syntax:
#   transpose_a=True      -> a "keyword argument": passing a value by its name,
#                            so the order of arguments doesn't matter
# dW2_manual = tf.matmul(h, probs-onehot, transpose_a=True) / 64

# --- Compare the two answers ---
# dW2_tape - dW2_manual -> subtract them, number by number (128 x 10 differences)
# np.abs(...)           -> make every difference positive (absolute value)
# .max()                -> a "method" call: take the biggest of those differences
# A tiny number (like 1e-08 = 0.00000001) means they match; it's just computer rounding.
# A big number would mean a bug somewhere.
# print(np.abs(dW2_tape - dW2_manual).max())

In [ ]:
lr = 0.1 # learning rate
def train_step(x,y):
    #gotta record the forward pass so we can compute the gradients
    with tf.GradientTape() as tape:
        loss = loss_fn(forward(x), y)
        # one gradient per parameter (W1, b1, W2, b2)
    grads = tape.gradient(loss, params)
    # zip pairs each parameter with its gradient, so we can update them together
    for param, gradients in zip(params, grads):
        # update the parameter by subtracting a fraction of its gradient param -= lr * gradients
        param.assign_sub(lr * gradients)
    return loss
# serve training data in shuffled batches of 64 images at a time
train_ds = (tf.data.Dataset.from_tensor_slices((x_train_flat, y_train)).shuffle(50000).batch(64))

history = {"loss":[], "train_acc":[],"val_acc":[]}
for epoch in range(10): # 10 passes through the training data
    losses = [train_step(xb, yb).numpy() for xb, yb in train_ds]
    history["loss"].append(np.mean(losses))
    history["train_acc"].append(accuracy_fn(x_train_flat, y_train))
    history['val_acc'].append(accuracy_fn(x_val_flat, y_val))
    print(f"Epoch {epoch}: loss={history['loss'][-1]:.4f}, train_acc={history['train_acc'][-1]:.4f}, val_acc={history['val_acc'][-1]:.4f}")

Epoch 0: loss=0.3947, train_acc=0.9280, val_acc=0.9232
Epoch 1: loss=0.2153, train_acc=0.9461, val_acc=0.9409
Epoch 2: loss=0.1642, train_acc=0.9617, val_acc=0.9547
Epoch 3: loss=0.1326, train_acc=0.9634, val_acc=0.9557
Epoch 4: loss=0.1120, train_acc=0.9719, val_acc=0.9616
Epoch 5: loss=0.0965, train_acc=0.9765, val_acc=0.9675
Epoch 6: loss=0.0842, train_acc=0.9800, val_acc=0.9693
Epoch 7: loss=0.0748, train_acc=0.9799, val_acc=0.9678
Epoch 8: loss=0.0667, train_acc=0.9840, val_acc=0.9701
Epoch 9: loss=0.0599, train_acc=0.9866, val_acc=0.9727
